# H2 — Choice Model: 3-Parameter Fit, Recovery, and Validation

**Prediction:** The 3-parameter model will capture choice with two recoverable per-subject parameters (k, beta) and vigor with a third (c_d).

**Sub-hypotheses:**
- **H2a:** Per-subject choice predictions will correlate r > 0.85 with observed choice proportions
- **H2b:** Parameter recovery will yield correlations r > 0.70 for k and beta in simulation
- **H2c:** k and beta will be approximately independent: |r| < 0.15
- **H2d:** The k x beta interaction in choice will be significant (mixed effects logistic)
- **H2e:** PPC 3x3 table — predicted vs observed choice surface

**What this determines:** Whether the computational model adequately captures individual differences in effort-threat integration, and whether the parameters are identifiable and meaningful.

In [ ]:
# ── Imports & Data Loading (self-contained) ──
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import pearsonr, zscore
from scipy.special import expit
from pathlib import Path

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10,
    'axes.spines.right': False, 'axes.spines.top': False,
})

DATA_DIR = Path("/workspace/data/exploratory_350/processed/stage5_filtered_data_20260320_191950")
OUT_DIR  = Path("/workspace/results/stats/full_analysis")
EXCLUDE  = [154, 197, 208]

# ── Choice trials ──
beh = pd.read_csv(DATA_DIR / "behavior.csv")
beh = beh[~beh['subj'].isin(EXCLUDE)].copy()
beh['T_round'] = beh['threat'].round(1)
beh['T_H'] = beh['distance_H'].map({1: 5.0, 2: 7.0, 3: 9.0})
beh['effort_reqT'] = beh['effort_H'] * beh['T_H'] - 0.4 * 5.0
beh['trial_number'] = beh.groupby('subj').cumcount() + 1
beh['current_score'] = beh.groupby('subj')['choice'].cumsum().shift(1, fill_value=0)

# ── Model parameters (pre-fitted — loading, not refitting) ──
params = pd.read_csv(OUT_DIR / "part1_params_full.csv")

print(f"Choice trials: {len(beh):,} ({beh['subj'].nunique()} subjects)")
print(f"Parameters:    {len(params)} subjects, columns = {list(params.columns)}")

## H2a — Per-subject choice prediction accuracy

Generate model predictions using fitted k and beta, compute per-subject correlation between predicted and observed P(heavy).  
**Test:** median r > 0.85 (exploratory: r = 0.985)

In [ ]:
# ── H2a: Per-subject choice prediction ──
# Choice equation: DeltaEU = 4 - k * effort(D) - beta * T
# P(heavy) = sigmoid(DeltaEU / tau)
# tau is population-level; use tau=1 for relative predictions (rank-preserving)

# Merge params into behavior
beh_p = beh.merge(params[['subj', 'k', 'beta']], on='subj')

# Model predictions per trial
beh_p['delta_eu'] = 4.0 - beh_p['k'] * beh_p['effort_reqT'] - beh_p['beta'] * beh_p['threat']
beh_p['p_heavy_pred'] = expit(beh_p['delta_eu'])  # tau=1 absorbed

# Per-subject: observed vs predicted P(heavy) across the 9 T x D conditions
subj_corrs = []
for subj, sg in beh_p.groupby('subj'):
    obs = sg.groupby(['T_round', 'distance_H'])['choice'].mean()
    pred = sg.groupby(['T_round', 'distance_H'])['p_heavy_pred'].mean()
    common = obs.index.intersection(pred.index)
    if len(common) >= 4:
        r, _ = pearsonr(obs.loc[common], pred.loc[common])
        subj_corrs.append({'subj': subj, 'r': r})

corr_df = pd.DataFrame(subj_corrs)
med_r = corr_df['r'].median()
mean_r = corr_df['r'].mean()

print("H2a — Per-subject choice prediction accuracy")
print("=" * 50)
print(f"  Median r  = {med_r:.3f}")
print(f"  Mean r    = {mean_r:.3f}")
print(f"  Min r     = {corr_df['r'].min():.3f}")
print(f"  N subjects = {len(corr_df)}")
print(f"\n  Verdict: {'PASS' if med_r > 0.85 else 'FAIL'} (threshold: r > 0.85)")

# Distribution plot
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(corr_df['r'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0.85, color='red', ls='--', label='Threshold (0.85)')
ax.axvline(med_r, color='orange', ls='-', lw=2, label=f'Median ({med_r:.3f})')
ax.set_xlabel('Per-subject r (observed vs predicted)')
ax.set_ylabel('Count')
ax.set_title('H2a: Choice Prediction Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

## H2b — Parameter recovery

Simulate data from known k,beta values, re-fit, and check recovery correlations.  
**Test:** r > 0.70 for both k and beta (exploratory: k = 0.860, beta = 0.883)

Note: Full recovery requires model fitting infrastructure. Here we do a lightweight simulation with N=50 synthetic subjects to demonstrate the approach.

In [ ]:
# ── H2b: Parameter recovery (lightweight MLE approach) ──
from scipy.optimize import minimize

# Design matrix: the 9 T x D conditions (5 trials each = 45 total)
threats   = np.array([0.1, 0.1, 0.1, 0.5, 0.5, 0.5, 0.9, 0.9, 0.9])
distances = np.array([1, 2, 3, 1, 2, 3, 1, 2, 3])
effort_map = {1: 0.6, 2: 0.8, 3: 1.0}
dist_map   = {1: 5.0, 2: 7.0, 3: 9.0}
effort_reqT = np.array([effort_map[d] * dist_map[d] - 0.4 * 5.0 for d in distances])
n_per_cond = 5

np.random.seed(42)
N_SIM = 50

# Draw true params from the empirical distribution
true_k    = np.exp(np.random.normal(-0.5, 0.7, N_SIM))
true_beta = np.exp(np.random.normal(1.7, 0.5, N_SIM))

def simulate_choices(k, beta, tau=1.0):
    delta_eu = 4.0 - k * effort_reqT - beta * threats
    p_heavy = expit(delta_eu / tau)
    choices = np.array([np.random.binomial(n_per_cond, p) for p in p_heavy])
    return choices

def neg_ll(params, choices):
    k, beta = np.exp(params[0]), np.exp(params[1])
    delta_eu = 4.0 - k * effort_reqT - beta * threats
    p = np.clip(expit(delta_eu), 1e-8, 1 - 1e-8)
    ll = np.sum(choices * np.log(p) + (n_per_cond - choices) * np.log(1 - p))
    return -ll

recovered_k, recovered_beta = [], []
for i in range(N_SIM):
    choices = simulate_choices(true_k[i], true_beta[i])
    res = minimize(neg_ll, [np.log(true_k[i]) + np.random.normal(0, 0.3),
                            np.log(true_beta[i]) + np.random.normal(0, 0.3)],
                   args=(choices,), method='Nelder-Mead')
    recovered_k.append(np.exp(res.x[0]))
    recovered_beta.append(np.exp(res.x[1]))

recovered_k = np.array(recovered_k)
recovered_beta = np.array(recovered_beta)

r_k, p_k = pearsonr(np.log(true_k), np.log(recovered_k))
r_b, p_b = pearsonr(np.log(true_beta), np.log(recovered_beta))

print("H2b — Parameter Recovery (N=50 simulated subjects)")
print("=" * 55)
print(f"  k recovery:    r = {r_k:.3f}, p = {p_k:.2e}  {'PASS' if r_k > 0.70 else 'FAIL'}")
print(f"  beta recovery: r = {r_b:.3f}, p = {p_b:.2e}  {'PASS' if r_b > 0.70 else 'FAIL'}")

# Scatter plots
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, true_v, rec_v, name, r_val in [
    (axes[0], np.log(true_k), np.log(recovered_k), 'log(k)', r_k),
    (axes[1], np.log(true_beta), np.log(recovered_beta), 'log(beta)', r_b)
]:
    ax.scatter(true_v, rec_v, alpha=0.5, s=20, color='steelblue')
    lims = [min(true_v.min(), rec_v.min()), max(true_v.max(), rec_v.max())]
    ax.plot(lims, lims, 'k--', alpha=0.3)
    ax.set_xlabel(f'True {name}')
    ax.set_ylabel(f'Recovered {name}')
    ax.set_title(f'{name}: r = {r_val:.3f}')
plt.suptitle('H2b: Parameter Recovery', y=1.02)
plt.tight_layout()
plt.show()

## H2c — k and beta orthogonality

**Test:** |r(log(k), log(beta))| < 0.15 (exploratory: r = 0.105)

In [ ]:
# ── H2c: k-beta orthogonality ──
r_kb, p_kb = pearsonr(params['log_k'], params['log_beta'])

print("H2c — k-beta Orthogonality")
print("=" * 45)
print(f"  r(log(k), log(beta)) = {r_kb:.3f}, p = {p_kb:.4f}")
print(f"  |r| = {abs(r_kb):.3f}")
print(f"\n  Verdict: {'PASS' if abs(r_kb) < 0.15 else 'FAIL'} (threshold: |r| < 0.15)")

# Scatter plot
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.scatter(params['log_k'], params['log_beta'], alpha=0.4, s=15, color='steelblue')
ax.set_xlabel('log(k) — effort sensitivity')
ax.set_ylabel('log(beta) — threat aversion')
ax.set_title(f'H2c: k-beta Independence (r = {r_kb:.3f})')
# Add regression line
m, b = np.polyfit(params['log_k'], params['log_beta'], 1)
x_line = np.linspace(params['log_k'].min(), params['log_k'].max(), 100)
ax.plot(x_line, m * x_line + b, 'r-', alpha=0.5, lw=1.5)
plt.tight_layout()
plt.show()

## H2d — k x beta interaction in choice

**Model:** `choice ~ k_z * beta_z + threat + distance + trial_number + current_score + (1 | subj)`  
**Test:** k_z:beta_z interaction is significant (exploratory: z = 7.23, p < .0001)

In [ ]:
# ── H2d: k x beta interaction ──
beh_int = beh.merge(params[['subj', 'log_k_z', 'log_beta_z']], on='subj')
beh_int.rename(columns={'log_k_z': 'k_z', 'log_beta_z': 'beta_z'}, inplace=True)

model_h2d = smf.logit(
    "choice ~ k_z * beta_z + threat + distance_H + trial_number + current_score",
    data=beh_int
).fit(disp=False, cov_type='cluster', cov_kwds={'groups': beh_int['subj']})

print("H2d — k x beta interaction in choice")
print("=" * 60)
print(model_h2d.summary2().tables[1][['Coef.', 'Std.Err.', 'z', 'P>|z|']].to_string())
print()

z_int = model_h2d.tvalues['k_z:beta_z']
p_int = model_h2d.pvalues['k_z:beta_z']
print(f"Interaction: z = {z_int:.3f}, p = {p_int:.2e}  {'PASS' if p_int < 0.05 else 'FAIL'}")

## H2e — PPC: Predicted vs Observed 3x3 Choice Surface

Posterior predictive check: compare the model-predicted and observed choice probability for each T x D condition.

In [ ]:
# ── H2e: PPC 3x3 table ──
obs_grid  = beh_p.groupby(['T_round', 'distance_H'])['choice'].mean().unstack()
pred_grid = beh_p.groupby(['T_round', 'distance_H'])['p_heavy_pred'].mean().unstack()

print("H2e — Posterior Predictive Check: 3x3 Choice Surface")
print("=" * 55)
print("\nObserved P(Heavy):")
print(obs_grid.round(3).to_string())
print("\nPredicted P(Heavy):")
print(pred_grid.round(3).to_string())
print("\nDifference (Pred - Obs):")
print((pred_grid - obs_grid).round(3).to_string())
print(f"\nMean absolute error: {(pred_grid - obs_grid).abs().values.mean():.4f}")

# Side-by-side heatmaps
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, data, title in [
    (axes[0], obs_grid, 'Observed'),
    (axes[1], pred_grid, 'Predicted'),
    (axes[2], pred_grid - obs_grid, 'Difference')
]:
    vmin, vmax = (0, 1) if 'Diff' not in title else (-0.15, 0.15)
    cmap = 'RdYlGn' if 'Diff' not in title else 'RdBu_r'
    im = ax.imshow(data.values, cmap=cmap, vmin=vmin, vmax=vmax,
                   origin='lower', aspect='auto')
    ax.set_xticks(range(3)); ax.set_xticklabels(['D=1', 'D=2', 'D=3'])
    ax.set_yticks(range(3)); ax.set_yticklabels(['T=0.1', 'T=0.5', 'T=0.9'])
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            val = data.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if abs(val) > 0.6 else 'black')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('H2e: PPC — Observed vs Predicted Choice Surface', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Parameter distribution plots ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, col, label, color in [
    (axes[0], 'log_k',    'log(k) — effort cost', 'steelblue'),
    (axes[1], 'log_beta', 'log(beta) — threat aversion', 'indianred'),
    (axes[2], 'log_cd',   'log(c_d) — capture aversion', 'seagreen')
]:
    vals = params[col].dropna()
    ax.hist(vals, bins=25, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(vals.median(), color='black', ls='--', lw=1.5, label=f'Median={vals.median():.2f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Parameter Distributions (N=290)', y=1.02)
plt.tight_layout()
plt.show()

## Summary

| Test | Prediction | Result | Verdict |
|------|-----------|--------|---------|
| H2a (choice accuracy) | r > 0.85 | _fill_ | _PASS/FAIL_ |
| H2b (k recovery) | r > 0.70 | _fill_ | _PASS/FAIL_ |
| H2b (beta recovery) | r > 0.70 | _fill_ | _PASS/FAIL_ |
| H2c (k-beta independence) | \|r\| < 0.15 | _fill_ | _PASS/FAIL_ |
| H2d (k x beta interaction) | p < 0.05 | _fill_ | _PASS/FAIL_ |
| H2e (PPC) | MAE < 0.10 | _fill_ | _qualitative_ |

**Interpretation:** If all pass, the 3-parameter model adequately captures individual differences in effort-threat integration. The k-beta orthogonality (H2c) and interaction (H2d) together show that effort and threat costs are separable dimensions that are genuinely integrated in choice.